In [0]:
# %pip install -r requirements.txt
%pip install -U diffusers transformers accelerate mlflow[databricks]==3.8.0

In [0]:
%restart_python

In [0]:
import os
import torch
from PIL import Image
from diffusers import QwenImageEditPlusPipeline

In [0]:
os.environ['HF_HOME'] = '/local_disk0/hf_home'
os.environ['HF_HUB_CACHE'] = '/local_disk0/models'

In [0]:
pipeline = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2509", 
                                                     torch_dtype=torch.bfloat16)
pipeline.to('cuda')
pipeline.set_progress_bar_config(disable=None)

print("pipeline loaded")

In [0]:
image1 = Image.open("sample_images/01_ice-castle-image.png").convert("RGB")
image2 = Image.open("sample_images/02_brown-bear-image.png").convert("RGB")
prompt = "The magician emperor bear is standing in front of castle with a diamond topped septar in his hand. Keep a snowing background and a blue sky. Do not include any bookmarks"
inputs = {
    "image": [image1, image2],
    "prompt": prompt,
    "generator": torch.manual_seed(0),
    "true_cfg_scale": 4.0,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "num_images_per_prompt": 1,
}
with torch.inference_mode():
    output = pipeline(**inputs)
    output_image = output.images[0]
    output_image.save("sample_images/output_image_edit_plus.png")
    print("image saved at", os.path.abspath("sample_images/output_image_edit_plus.png"))

In [0]:
%sh
nvidia-smi